# Fine-tuning the Model

通常有三种应用for LLM
1. Continue pretrain 
2. Fine tunning
3. For the preference

 
## Step
1. Full tuning / LoRA / other PEFT methods  
2. Data preparation  
   - Instruction  
   - Prompt-completion  
   - Conversational data  
3. Trainer selection  
   - `transformers.Trainer`  
   - `trl.SFTTrainer`  
4. Train the model  
5. Save the whole model or adapter  
6. Deploy the tuned model for inference  

---

# 1. Fine-tuning Methods

## 1.1 Full Fine-tuning
Update **all model parameters**.

### Features
- Strong adaptation ability  
- Expensive in GPU memory and training cost  
- For each task, usually need one full checkpoint  

### Good for
- Small / medium models  
- Enough GPU resources  
- Large dataset  
- Want maximum task adaptation  

---

## 1.2 PEFT: Parameter-Efficient Fine-Tuning
Do **not** update all parameters.  
Freeze most of the base model and train only a small number of parameters.

---

   ### 1.3 Adapter Tuning
   Insert small trainable modules into the network.

   #### Idea
   - Base model frozen  
   - Only adapter modules are trained  

   #### Pros
   - Small task-specific checkpoints  
   - Easy to switch tasks  

   #### Cons
   - Need extra modules in inference  
   - Sometimes less standard than LoRA in current LLM workflows  


      ### 1.31 LoRA
      Core idea:

      \[
      y = (W + \Delta W)x, \qquad \Delta W = BA
      \]

      where:
      - `W`: pretrained weight, usually frozen  
      - `A, B`: small trainable low-rank matrices  
      - rank `r` is much smaller than full dimension  

      #### Meaning
      Instead of learning a full large update matrix, LoRA learns a **low-rank approximation** of the update.

      #### Important correction
      LoRA is **not only for MLP layers**.  
      It is most commonly applied to **linear layers**, especially in Transformer:
      - `q_proj`
      - `k_proj`
      - `v_proj`
      - `o_proj`

      It may also be applied to:
      - MLP projection layers
      - other linear modules

      #### Advantages
      - Faster training  
      - Much fewer trainable parameters  
      - Easy to save one adapter per downstream task  
      - Can merge into the base model later  

      #### Disadvantages
      - Choice of `target_modules` matters  
      - Rank `r` must be tuned  
      - Sometimes may underperform full fine-tuning  


      ### 1.32 QLoRA
      QLoRA = **quantized base model + LoRA**

      #### Idea
      - Load pretrained model in **4-bit**
      - Freeze quantized base model
      - Train LoRA adapters on top

      #### Why
      Reduce GPU memory usage while keeping adaptation ability.

      #### Important
      QLoRA does **not** mean “train the whole quantized model.”  
      Usually:
      - base model frozen
      - only LoRA parameters are trainable

---

## 1.4 Soft Prompt Approaches
These methods do not mainly change model weights; they train prompt-related parameters.

   ### Prompt Tuning
   Train a set of **virtual prompt embeddings** added before input tokens.

   ### Prefix Tuning
   Train prefix representations injected into **each layer’s attention key/value**.

   ### P-tuning / P-tuning v2
   More flexible prompt parameterization.  
   Often deeper / more expressive than simple prompt tuning.

----- 

# 2. SFT Training

## 2.1 What is SFT
SFT = **Supervised Fine-Tuning**

Train on labeled pairs:

- input → output
- prompt → completion
- messages → assistant response

### Important
SFT is a **training objective / paradigm**.  
It does **not** mean full parameter update.

So:
- Full fine-tuning can do SFT
- LoRA can do SFT
- QLoRA can do SFT

---

## 2.2 Instruction Tuning is usually a common form of SFT.

Typical examples:
- single instruction + answer
- multi-turn assistant conversation
- prompt-completion tasks

So in practice:
**Instruction tuning ≈ SFT on instruction-following data**

### SFT can do
- You can do SFT with full fine-tuning
- You can do SFT with LoRA
- You can also do instruction-style adaptation with prompt tuning

---

# 3. Where SFT Sits in the Alignment Pipeline 


## 3.1 Pretraining
The model first learns:
- language patterns
- world knowledge
- general next-token prediction ability

But after pretraining, it may still:
- not follow instructions well
- not answer in assistant style
- not output safe / preferred responses
- not follow formatting reliably

---

## 3.2 Why SFT is called “the first step of alignment”
SFT teaches the model:
- follow instructions
- answer in assistant style
- respect role structure (`system / user / assistant`)
- produce better formatted outputs

So you can say:

**SFT is often the first step of task adaptation and behavioral alignment after pretraining.** 




---

## 3.3 But SFT is not the whole alignment story
After SFT, people may continue with preference-based alignment: 人类更喜欢什么答案”来继续优化模型 to Make responses more aligned with human preferences. 

 - Reinforcement Learning from Human Feedback

      #### Idea
      1. Collect multiple candidate answers  
      2. Humans rank them  
      3. Train a reward model  
      4. Optimize the model to get better rewards  

- DPO: Direct Preference Optimization, simpler than RLHF, more stable 

      #### Idea
      Use preference pairs directly:
      - chosen response
      - rejected response
      No separate RL loop is needed.

---

### Preference Optimization ( Code how. the do )
A bigger category meaning:
- optimize the model using human preference signals

Includes methods like:
- RLHF
- DPO
- related preference-based objectives

---

## 3.4 Simple chain
Pretraining  
→ SFT  
→ Preference alignment (RLHF / DPO / similar methods)

So:

- **Pretraining**: learn language + knowledge  
- **SFT**: learn instruction-following behavior  
- **RLHF / DPO**: refine behavior by preference  


TRL 官方文档说明，SFTTrainer 支持 language modeling、prompt-completion 和 conversational dataset；对 conversational dataset 会自动应用 chat template。 
1. pure text  {"text": ...} for pretraineing : domain adaptation
2. prompt completion { "prompt" : " ", "completion": ....}: 适合单轮指令任务 
3. converation: {"message":[ {"role": , "content":... }]}: chat , assistant, multiple round . 


---


# 4. Formulate the Data

## 4.1 Why data reformulation is needed
The original dataset may not already be in the exact format the model expects.

The formatting tells the model:
- where instruction starts
- where answer starts
- what role each message belongs to
- where EOS should be
- which part should be predicted

---

## 4.2 Original raw instruction format
Example:

```json
{
  "instruction": "Explain LoRA",
  "input": "",
  "output": "LoRA is a low-rank adaptation method."
} 


API will do the COT, prompt instruction, RAG, call function 
LLM local model usually  do the fine tunning of the model  

## Fine tuning model style 
Full fine tunining code 


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments, # training argument 
    Trainer, #For training the model 
    DataCollatorForLanguageModeling,# collact in to batch data 
) 

# get the data
# tokenize function that apply for one sample
# apply on the data
# get the model 
# get the training argument from TrainingArguments 
# get the peft model if use the lora other wise 
# Trainer ( model, args , data )

model_id = "gpt2"

# the text form that can train the LLM from transforemr 
data = [
    {"text": "### Instruction:\nExplain LoRA.\n\n### Response:\nLoRA is a low-rank adaptation method."},
    {"text": "### Instruction:\nWhat is QLoRA?\n\n### Response:\nQLoRA combines 4-bit quantization with LoRA adapters."},
] 


# formulate the data  by the Dataset 
dataset = Dataset.from_list( data )


tokenizer = AutoTokenizer.from_pretrained( model_id)
tokenizer.pad_token = tokenizer.eos_token  

# how to determine these paarameters 
def tokenize_fn( sample) : 
    out = tokenizer( 
                    
            sample['text'],  # data 
            truncation = True, 
            padding = 'max_length', 
            max_length = 256, 
                    )

    out['label'] = out['input_ids'].copy()
    
    return out 


tokenized_dataset = dataset.map( tokenize_fn )
model = AutoModelForCasualLM.from_pretrained(model_id) 


args = TrainingArguments(
    output_dir="./full_ft_out",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=50,
    fp16=False,
)


trainer = Trainer( 
                  model= model, 
                  args = args,
                  train_dataset = tokenized_dataset, 
                  data_collator = DataCollatorForLauguageModeling( 
                    tokenizer = tokenizer, mlm= False ))

# {
#   "input_ids": tensor(batch_size, seq_len),
#    "attention_mask": tensor(batch_size, seq_len),
#    "labels": tensor(batch_size, seq_len)
# }   data_collacator collect the data into batches.  用 tokenizer 帮忙处理 padding 等事情 
# Train the model 

trainer.train() 

model.save_pretrained(f"./full_ft_out/final_model")
tokenizer.save_pretrained( f"./full_ft_out/final_model" )

# tokenizer save : tokenizer.json, config, special token maps, vocab json .. 
# model save the config, generation config, model safetensors. 


In [ ]:
# Load the mode 

## Fine tunning model style 

LoRA  + SFT trainer  for instruction tunning 

needs to import trl and the peft 

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer 

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


model_id = "distilgpt2"

# the data is a list, adding different {} dict.
# No need to tokenize the data separately 
# define the model 
# define the lora config 
# SFTConfig store the training argument 
# the trainer from SFTtrainer include model, args, peft_config, processing_class = tokenizer, train_dataset (not tokenized, the original formated text). 
# transfomer Trainer has model, args,train_dataset. This training data is tokenized. 


data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful LLM tutor."},
            {"role": "user", "content": "Explain LoRA."},
            {"role": "assistant", "content": "LoRA is a parameter-efficient fine-tuning method using low-rank adapters."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful LLM tutor."},
            {"role": "user", "content": "Explain QLoRA."},
            {"role": "assistant", "content": "QLoRA uses a 4-bit quantized base model and trains LoRA adapters on top."},
        ]
    },
] 

train_dataset  = Dataset.from_list(data) 

tokenizer= AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# not tokenizer the data 

model = AutoModelForCausalLM.from_pretrained(model_id)

peft_config = LoraConfig(
    r = 8,
    lora_alpha = 16, 
    lora_dropout = 0.05, 
    bias = 'none',
    task_type= '"CASUAL_LM', 
    target_modules = ['c_attn'] # target model needs to be checked for the specific model name 
    
)


#Training arugment is set with the SFT 
training_args = SFTConfig( 
        output_dir = './lora_sft_out', 
        per_device_train_batch_size = 2, 
        num_train_epochs = 3, 
        learning_rate = 2e-4,
        logging_step =40, 
        save_steps = 50, 
        max_seq_length =256, 
          
                        
                          )

trainer = SFTTrainer( 
            model = model,
            args =training_args,
            train_dataset = train_dataset,
            processing_class=tokenizer,
            peft_config = peft_config)


trainer.train()
trainer.model.save_pretrained( "./lora_sft_out/final_adapter")
tokenizer.save_pretrained( "./lora_sft_out/final_adapter")
 

In [ ]:
# this form is to save the model and then load the whole model 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "facebook/opt-350m"
adapter_path = "./qlora_out/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, adapter_path)

prompt = "Explain PEFT in one paragraph."
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)) 


# QLoRA + SFTTrainer 


In [ ]:
from peft import prepare_model_for_kbit_training

from transformers import BitsAndBytesConfig 

model_id = "facebook/opt-350m"

data = [
    {
        "messages": [
            {"role": "system", "content": "You are an expert teacher."},
            {"role": "user", "content": "What is instruction tuning?"},
            {"role": "assistant", "content": "Instruction tuning is supervised fine-tuning on instruction-response data."},
        ]
        
    },
    {
        "messages": [
            {"role": "system", "content": "You are an expert teacher."},
            {"role": "user", "content": "What is PEFT?"},
            {"role": "assistant", "content": "PEFT adapts a model by training only a small number of parameters."},
        ]
    },
]


train_dataset = Dataset.from_list(data)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token 

# bnb config is need load_in_4bit, 
bnb_config = BitsAndBytesConfig(   
                    load_in_4bit = True,
                    bnb_4bit_quant_type = 'nf4',
                    bnb_4bit_use_double_quant = True,
                    bnb_4bit_compute_type = torch.bfloat16 if torch.cuda.is_avaliable() else torch.float32)


# quantization, device  
model = AutoModelForCasualLM.from_pretrained( 
                        model_id, 
                        quantatization_config = bnb_config, 
                        device_map = 'auto')
  
  
# for training stable, use the 
  
model = prepare_model_for_kbit_training(model)



peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # 按具体模型调整
)


args = SFTConfig( 
    output_dir="./qlora_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    max_seq_length=256,
    bf16=torch.cuda.is_available(),
)
                 
    


trainer = SFTTrainer(
    model = model, 
    args = args,
    train_dataset = train_dataset,
    processing_class = tokenizer, 
    peft_config = peft_config
    

 )



trainer.train()
trainer.model.save_pretrained("./qlora_out/final_adapter")
tokenizer.save_pretrained("./qlora_out/final_adapter") 



 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "facebook/opt-350m"
adapter_path = "./my_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, adapter_path)
# save the whole model  

merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")  



#if merged_model = trainer.model.merge_and_unload()   it will directly save the merge and unload model  

# Data transform the data form text to the message styles 



In [ ]:
raw_data = [
    {
        "instruction": "Explain the difference between LoRA and QLoRA.",
        "input": "",
        "output": "LoRA trains low-rank adapters on a full-precision base model, while QLoRA trains LoRA adapters on a 4-bit quantized base model."
    },
    {
        "instruction": "What is PEFT?",
        "input": "",
        "output": "PEFT stands for Parameter-Efficient Fine-Tuning."
    },
] 

 
formatted_data = []

for ex in raw_data:
    user_content = ex["instruction"]
    if ex["input"]:
        user_content += f"\n\nInput:\n{ex['input']}"

    formatted_data.append(
        {
            "messages": [
                {"role": "system", "content": "You are a helpful expert in LLM fine-tuning."},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": ex["output"]},
            ]
        }
    )

print(formatted_data[0])



### Preference Optimization ( Code how. the do )  